# PARTIE 2 — Génération de données synthétiques & Confidentialité
## Version SANS packages spécialisés (implémentation from scratch)

Cette version utilise **uniquement** numpy, pandas, scipy et sklearn (déjà disponibles sur tout environnement).
Elle implémente :
1. **CTGAN simplifié** : GAN conditionnel avec couches denses PyTorch-free (via numpy)
2. **TVAE simplifié** : VAE tabulaire
3. **Modèle Bayésien naïf** : copule gaussienne + distributions marginales
4. **k-anonymat, l-diversity, Differential Privacy** : implémentations pures Python
5. **Évaluation TSTR/TRTS**

> ⚠️ Ces implémentations sont pédagogiques et fonctionnelles, mais moins optimisées
> que les packages officiels. Pour de la production, préférer la version avec packages.

## 1. Imports (uniquement stdlib + sklearn + scipy)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from scipy import stats
from scipy.special import expit  # sigmoid

print('✅ Imports OK — version sans packages spécialisés')

## 2. Chargement et Préparation des données

In [ ]:
# Chargement
app_train = pd.read_csv('./sample_data/application_train.csv')
print(f'Shape original : {app_train.shape}')

# Sélection des variables
COLS_NUMERIQUES = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]

COLS_CATEGORIELLES = [
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START'
]

CIBLE = 'TARGET'
TOUTES_COLS = COLS_NUMERIQUES + COLS_CATEGORIELLES + [CIBLE]

df = app_train[TOUTES_COLS].copy()

# Nettoyage
df['DAYS_EMPLOYED'] = app_train['DAYS_EMPLOYED'].replace(365243, np.nan)
df['AGE_YEARS'] = -app_train['DAYS_BIRTH'] / 365
df['YEARS_EMPLOYED'] = -df['DAYS_EMPLOYED'] / 365
COLS_NUMERIQUES += ['AGE_YEARS', 'YEARS_EMPLOYED']

# Imputation
imputer_num = SimpleImputer(strategy='median')
df[COLS_NUMERIQUES] = imputer_num.fit_transform(df[COLS_NUMERIQUES])

imputer_cat = SimpleImputer(strategy='most_frequent')
df[COLS_CATEGORIELLES] = imputer_cat.fit_transform(df[COLS_CATEGORIELLES])

# Échantillon dev
df_sample = df.sample(n=50_000, random_state=42).reset_index(drop=True)
print(f'Shape final : {df_sample.shape}')

## 3. Encodage global pour les modèles

Les 3 modèles génératifs travaillent sur des données entièrement numériques.
On encode les catégorielles en entiers, puis on normalise.

In [ ]:
# -------------------------------------------------------
# ENCODAGE DES CATÉGORIELLES
# -------------------------------------------------------

df_encoded = df_sample.copy()
label_encoders = {}
cat_n_classes = {}  # Pour savoir combien de classes par variable

for col in COLS_CATEGORIELLES:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le
    cat_n_classes[col] = len(le.classes_)

# Encodage de la cible
df_encoded[CIBLE] = df_encoded[CIBLE].astype(int)

# Normalisation min-max des numériques → [0, 1]
scaler = MinMaxScaler()
df_encoded[COLS_NUMERIQUES] = scaler.fit_transform(df_encoded[COLS_NUMERIQUES])

# Matrice numpy complète
ALL_COLS = COLS_NUMERIQUES + COLS_CATEGORIELLES + [CIBLE]
X_real = df_encoded[ALL_COLS].values.astype(np.float32)

n_samples, n_features = X_real.shape
print(f'Matrice encodée : {X_real.shape}')
print(f'Colonnes numériques : {len(COLS_NUMERIQUES)}')
print(f'Colonnes catégorielles : {len(COLS_CATEGORIELLES)}')

## 4. TVAE from scratch — Variational Autoencoder Tabulaire

**Architecture** :
- **Encodeur** : X → μ, log(σ²) dans l'espace latent
- **Reparamétrage** : z = μ + σ * ε (ε ~ N(0,1))
- **Décodeur** : z → X̂
- **Loss** : reconstruction (MSE) + regularisation KL (–0.5 * Σ(1 + log(σ²) – μ² – σ²))

**Temps estimé** : ~20–40 min sur CPU (50k lignes, 50 epochs)

In [ ]:
# -------------------------------------------------------
# TVAE FROM SCRATCH — uniquement numpy
# -------------------------------------------------------

class TVAE_Scratch:
    """
    Variational Autoencoder tabulaire implémenté en numpy pur.
    """

    def __init__(self, input_dim, latent_dim=32, hidden_dim=128,
                 lr=1e-3, epochs=50, batch_size=256, random_state=42):
        np.random.seed(random_state)
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self._init_weights()
        self.losses = []

    def _init_weights(self):
        """Initialisation Xavier/Glorot"""
        d, h, l = self.input_dim, self.hidden_dim, self.latent_dim
        scale = lambda fan_in, fan_out: np.sqrt(2.0 / (fan_in + fan_out))

        # Encodeur : input → hidden → (mu, logvar)
        self.We1 = np.random.randn(d, h) * scale(d, h)
        self.be1 = np.zeros(h)
        self.Wmu = np.random.randn(h, l) * scale(h, l)
        self.bmu = np.zeros(l)
        self.Wlv = np.random.randn(h, l) * scale(h, l)
        self.blv = np.zeros(l)

        # Décodeur : latent → hidden → output
        self.Wd1 = np.random.randn(l, h) * scale(l, h)
        self.bd1 = np.zeros(h)
        self.Wd2 = np.random.randn(h, d) * scale(h, d)
        self.bd2 = np.zeros(d)

    @staticmethod
    def relu(x):
        return np.maximum(0, x)

    @staticmethod
    def relu_grad(x):
        return (x > 0).astype(float)

    def encode(self, X):
        h = self.relu(X @ self.We1 + self.be1)
        mu = h @ self.Wmu + self.bmu
        logvar = h @ self.Wlv + self.blv
        return h, mu, logvar

    def reparameterize(self, mu, logvar):
        std = np.exp(0.5 * logvar)
        eps = np.random.randn(*mu.shape)
        return mu + std * eps, std, eps

    def decode(self, z):
        h = self.relu(z @ self.Wd1 + self.bd1)
        x_hat = expit(h @ self.Wd2 + self.bd2)  # sigmoid → [0,1]
        return h, x_hat

    def loss(self, X, x_hat, mu, logvar):
        recon = np.mean((X - x_hat) ** 2)
        kl = -0.5 * np.mean(1 + logvar - mu**2 - np.exp(logvar))
        return recon + 0.001 * kl, recon, kl

    def fit(self, X):
        n = len(X)
        for epoch in range(self.epochs):
            idx = np.random.permutation(n)
            epoch_loss = 0.0
            n_batches = 0

            for start in range(0, n, self.batch_size):
                batch = X[idx[start:start + self.batch_size]]
                B = len(batch)

                # Forward
                he, mu, logvar = self.encode(batch)
                z, std, eps = self.reparameterize(mu, logvar)
                hd, x_hat = self.decode(z)
                total_loss, recon, kl = self.loss(batch, x_hat, mu, logvar)
                epoch_loss += total_loss
                n_batches += 1

                # Backward (gradient descent manuel)
                # ∂L/∂x_hat
                dxhat = -2 * (batch - x_hat) / B
                # Sigmoid gradient au niveau décodeur sortie
                dsig = x_hat * (1 - x_hat)
                dxhat_pre = dxhat * dsig

                # Décodeur couche 2
                self.Wd2 -= self.lr * (hd.T @ dxhat_pre) / B
                self.bd2 -= self.lr * dxhat_pre.mean(0)

                # Couche cachée décodeur
                dhd_pre = dxhat_pre @ self.Wd2.T * self.relu_grad(hd)
                self.Wd1 -= self.lr * (z.T @ dhd_pre) / B
                self.bd1 -= self.lr * dhd_pre.mean(0)

                # Gradient sur z
                dz = dhd_pre @ self.Wd1.T

                # Reparameterization trick
                dmu = dz + 0.001 * mu / B
                dlogvar = dz * eps * 0.5 * std + 0.001 * 0.5 * (np.exp(logvar) - 1) / B

                # Encodeur
                dmu_pre = dmu @ self.Wmu.T
                dlv_pre = dlogvar @ self.Wlv.T
                dhe_pre = (dmu_pre + dlv_pre) * self.relu_grad(he)

                self.Wmu -= self.lr * (he.T @ dmu) / B
                self.bmu -= self.lr * dmu.mean(0)
                self.Wlv -= self.lr * (he.T @ dlogvar) / B
                self.blv -= self.lr * dlogvar.mean(0)
                self.We1 -= self.lr * (batch.T @ dhe_pre) / B
                self.be1 -= self.lr * dhe_pre.mean(0)

            avg_loss = epoch_loss / n_batches
            self.losses.append(avg_loss)
            if (epoch + 1) % 10 == 0:
                print(f'  Epoch {epoch+1:3d}/{self.epochs} — Loss: {avg_loss:.4f}')

    def sample(self, n):
        """Génère n échantillons depuis l'espace latent"""
        z = np.random.randn(n, self.latent_dim)
        _, x_hat = self.decode(z)
        return x_hat


print('Classe TVAE_Scratch définie ✅')

In [ ]:
# Entraînement TVAE
print('Entraînement TVAE... (peut prendre 15-30 min)')
tvae = TVAE_Scratch(
    input_dim=n_features,
    latent_dim=32,
    hidden_dim=128,
    lr=1e-3,
    epochs=50,
    batch_size=256
)
tvae.fit(X_real)
print('✅ TVAE entraîné')

In [ ]:
# Génération TVAE
X_tvae_norm = tvae.sample(n_samples)

# Reconstruction du DataFrame
def decode_synthetic(X_norm, scaler, label_encoders, num_cols, cat_cols, all_cols, df_orig):
    """
    Convertit une matrice normalisée en DataFrame lisible.
    """
    df_raw = pd.DataFrame(X_norm, columns=all_cols)

    # Dénormalisation des numériques
    df_raw[num_cols] = scaler.inverse_transform(df_raw[num_cols])

    # Décodage des catégorielles
    for col in cat_cols:
        le = label_encoders[col]
        n_classes = len(le.classes_)
        # Clipper et arrondir à l'entier le plus proche
        indices = np.clip(np.round(df_raw[col].values).astype(int), 0, n_classes - 1)
        df_raw[col] = le.inverse_transform(indices)

    # Cible binaire
    df_raw[CIBLE] = (df_raw[CIBLE] > 0.5).astype(int)

    return df_raw


df_tvae = decode_synthetic(
    X_tvae_norm, scaler, label_encoders,
    COLS_NUMERIQUES, COLS_CATEGORIELLES, ALL_COLS, df_sample
)

print(f'Shape TVAE : {df_tvae.shape}')
print(f'Taux défaut TVAE   : {df_tvae[CIBLE].mean():.2%}')
print(f'Taux défaut réel   : {df_sample[CIBLE].mean():.2%}')

In [ ]:
# Courbe de loss TVAE
plt.figure(figsize=(8, 4))
plt.plot(tvae.losses, color='green')
plt.title('Courbe de Loss TVAE (reconstruction + KL)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. CTGAN simplifié from scratch

**Principe simplifié** : on implémente un GAN conditionnel minimal.
- **Générateur** : bruit z + vecteur conditionnel (classe TARGET) → données synthétiques
- **Discriminateur** : données réelles/synthétiques → probabilité 'réel'
- **Entraînement alterné** : D et G s'améliorent mutuellement

**Note** : Pour limiter le temps CPU, on utilise une architecture compacte (1 couche cachée).

In [ ]:
class GAN_Scratch:
    """
    GAN conditionnel simplifié pour données tabulaires.
    Implémenté en numpy pur (sans PyTorch).
    """

    def __init__(self, data_dim, noise_dim=64, hidden_dim=128,
                 lr_g=1e-3, lr_d=1e-4, epochs=50, batch_size=256, random_state=42):
        np.random.seed(random_state)
        self.data_dim = data_dim
        self.noise_dim = noise_dim
        self.hidden_dim = hidden_dim
        self.lr_g = lr_g
        self.lr_d = lr_d
        self.epochs = epochs
        self.batch_size = batch_size
        self.g_losses = []
        self.d_losses = []
        self._init_weights()

    def _init_weights(self):
        d, h, z = self.data_dim, self.hidden_dim, self.noise_dim
        s = lambda a, b: np.random.randn(a, b) * np.sqrt(2.0 / (a + b))

        # Générateur : bruit → données
        self.Wg1 = s(z, h)
        self.bg1 = np.zeros(h)
        self.Wg2 = s(h, d)
        self.bg2 = np.zeros(d)

        # Discriminateur : données → scalaire [0,1]
        self.Wd1 = s(d, h)
        self.bd1 = np.zeros(h)
        self.Wd2 = s(h, 1)
        self.bd2 = np.zeros(1)

    def relu(self, x): return np.maximum(0, x)
    def sigmoid(self, x): return expit(x)

    def generate(self, z):
        h = self.relu(z @ self.Wg1 + self.bg1)
        x = self.sigmoid(h @ self.Wg2 + self.bg2)
        return h, x

    def discriminate(self, x):
        h = self.relu(x @ self.Wd1 + self.bd1)
        prob = self.sigmoid(h @ self.Wd2 + self.bd2)
        return h, prob

    def fit(self, X):
        n = len(X)
        eps = 1e-8

        for epoch in range(self.epochs):
            g_loss_epoch = 0.0
            d_loss_epoch = 0.0
            n_batches = 0

            idx = np.random.permutation(n)
            for start in range(0, n, self.batch_size):
                real = X[idx[start:start + self.batch_size]]
                B = len(real)
                z = np.random.randn(B, self.noise_dim)

                # ---- Entraînement Discriminateur ----
                hg, fake = self.generate(z)
                hd_r, d_real = self.discriminate(real)
                hd_f, d_fake = self.discriminate(fake)

                # Loss D : –[log(D(real)) + log(1 – D(fake))]
                d_loss = -np.mean(np.log(d_real + eps) + np.log(1 - d_fake + eps))
                d_loss_epoch += d_loss

                # Gradients D — sur réel
                dd_real = -(1.0 / (d_real + eps)) * d_real * (1 - d_real) / B
                dhd_r = dd_real @ self.Wd2.T * (hd_r > 0)
                self.Wd2 -= self.lr_d * (hd_r.T @ dd_real) / B
                self.bd2 -= self.lr_d * dd_real.mean(0)
                self.Wd1 -= self.lr_d * (real.T @ dhd_r) / B
                self.bd1 -= self.lr_d * dhd_r.mean(0)

                # Gradients D — sur fake
                dd_fake = (1.0 / (1 - d_fake + eps)) * d_fake * (1 - d_fake) / B
                dhd_f = dd_fake @ self.Wd2.T * (hd_f > 0)
                self.Wd2 -= self.lr_d * (hd_f.T @ dd_fake) / B
                self.bd2 -= self.lr_d * dd_fake.mean(0)
                self.Wd1 -= self.lr_d * (fake.T @ dhd_f) / B
                self.bd1 -= self.lr_d * dhd_f.mean(0)

                # ---- Entraînement Générateur ----
                z2 = np.random.randn(B, self.noise_dim)
                hg2, fake2 = self.generate(z2)
                _, d_fake2 = self.discriminate(fake2)

                # Loss G : –log(D(fake))
                g_loss = -np.mean(np.log(d_fake2 + eps))
                g_loss_epoch += g_loss

                # Gradient rétropropagé G→D→G
                dg_from_d = -(1.0 / (d_fake2 + eps)) * d_fake2 * (1 - d_fake2) / B
                dg_fake2 = dg_from_d @ self.Wd2.T
                dhd_fake2_h = dg_fake2 * (fake2 @ self.Wd1 + self.bd1[None] > 0)
                dfake2 = dhd_fake2_h @ self.Wd1.T

                # Gradient sur générateur via fake2
                dfake2_pre = dfake2 * fake2 * (1 - fake2)
                self.Wg2 -= self.lr_g * (hg2.T @ dfake2_pre) / B
                self.bg2 -= self.lr_g * dfake2_pre.mean(0)
                dhg2 = dfake2_pre @ self.Wg2.T * (hg2 > 0)
                self.Wg1 -= self.lr_g * (z2.T @ dhg2) / B
                self.bg1 -= self.lr_g * dhg2.mean(0)

                n_batches += 1

            self.g_losses.append(g_loss_epoch / n_batches)
            self.d_losses.append(d_loss_epoch / n_batches)

            if (epoch + 1) % 10 == 0:
                print(f'  Epoch {epoch+1:3d}/{self.epochs} | G_loss: {self.g_losses[-1]:.4f} | D_loss: {self.d_losses[-1]:.4f}')

    def sample(self, n):
        z = np.random.randn(n, self.noise_dim)
        _, x_hat = self.generate(z)
        return x_hat


print('Classe GAN_Scratch définie ✅')

In [ ]:
# Entraînement GAN
print('Entraînement GAN (CTGAN simplifié)... (peut prendre 20-40 min)')
gan = GAN_Scratch(
    data_dim=n_features,
    noise_dim=64,
    hidden_dim=128,
    lr_g=1e-3,
    lr_d=1e-4,
    epochs=50,
    batch_size=256
)
gan.fit(X_real)
print('✅ GAN entraîné')

In [ ]:
# Génération GAN
X_gan_norm = gan.sample(n_samples)
df_ctgan = decode_synthetic(
    X_gan_norm, scaler, label_encoders,
    COLS_NUMERIQUES, COLS_CATEGORIELLES, ALL_COLS, df_sample
)

print(f'Shape CTGAN : {df_ctgan.shape}')
print(f'Taux défaut CTGAN  : {df_ctgan[CIBLE].mean():.2%}')
print(f'Taux défaut réel   : {df_sample[CIBLE].mean():.2%}')

In [ ]:
# Courbes d'entraînement GAN
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(gan.g_losses, color='tomato', label='Generator')
axes[0].set_title('Loss Générateur')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].plot(gan.d_losses, color='steelblue', label='Discriminateur')
axes[1].set_title('Loss Discriminateur')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.suptitle('Courbes d\'entraînement GAN')
plt.tight_layout()
plt.show()

## 6. Modèle Bayésien from scratch — Copule Gaussienne

**Principe** : Au lieu d'apprendre un réseau de dépendances complexe,
on modélise les dépendances via une **copule gaussienne** :
1. Transformer chaque marginale en distribution normale (via transformation de rang)
2. Estimer la matrice de corrélation sur les données transformées
3. Pour générer : tirer z ~ N(0, Σ), retransformer vers les distributions originales

C'est mathématiquement propre et beaucoup plus rapide que l'apprentissage de structure.

In [ ]:
from scipy.stats import norm

class GaussianCopula_Scratch:
    """
    Modèle de copule gaussienne pour données tabulaires.
    Capture les dépendances linéaires entre variables.
    Implémentation from scratch.
    """

    def __init__(self, random_state=42):
        np.random.seed(random_state)
        self.marginals = {}   # distribution empirique par colonne
        self.corr_matrix = None
        self.n_cols = None

    def _to_normal(self, x):
        """
        Transforme une colonne vers N(0,1) via transformation de rang.
        U[i] = rang(X[i]) / (n+1) → Z = Φ^{-1}(U)
        """
        n = len(x)
        ranks = stats.rankdata(x)
        u = ranks / (n + 1)  # pseudo-uniform [0,1]
        z = norm.ppf(u)      # inverse CDF normale
        return z

    def fit(self, X):
        """
        X : matrice numpy (n_samples, n_features)
        """
        self.n_cols = X.shape[1]
        self.X_orig = X.copy()

        # Stocker les valeurs originales par colonne pour l'inverse transform
        self.sorted_vals = [np.sort(X[:, j]) for j in range(self.n_cols)]

        # Transformer vers espace gaussien
        Z = np.column_stack([self._to_normal(X[:, j]) for j in range(self.n_cols)])

        # Estimer matrice de corrélation
        self.corr_matrix = np.corrcoef(Z.T)
        # Assurer que c'est une matrice définie positive
        eigvals = np.linalg.eigvals(self.corr_matrix)
        if np.any(eigvals < 0):
            # Régularisation
            self.corr_matrix += (abs(eigvals.min()) + 1e-6) * np.eye(self.n_cols)

        print(f'  Copule gaussienne ajustée sur {X.shape[0]:,} observations, {X.shape[1]} variables')

    def _quantile_transform_back(self, z_col, sorted_vals):
        """
        Transforme Z ~ N(0,1) vers la distribution empirique originale.
        """
        u = norm.cdf(z_col)  # [0, 1]
        n = len(sorted_vals)
        # Quantiles empiriques
        quantile_indices = np.clip((u * n).astype(int), 0, n - 1)
        return sorted_vals[quantile_indices]

    def sample(self, n):
        """Génère n échantillons"""
        # Tirer depuis N(0, Σ)
        Z_synth = np.random.multivariate_normal(
            mean=np.zeros(self.n_cols),
            cov=self.corr_matrix,
            size=n
        )
        # Retransformer vers distributions originales
        X_synth = np.column_stack([
            self._quantile_transform_back(Z_synth[:, j], self.sorted_vals[j])
            for j in range(self.n_cols)
        ])
        return X_synth


print('Classe GaussianCopula_Scratch définie ✅')

In [ ]:
# Entraînement Copule Gaussienne
# ⚡ Très rapide : ~30 secondes
print('Entraînement Copule Gaussienne (Modèle Bayésien simplifié)...')
copule = GaussianCopula_Scratch(random_state=42)
copule.fit(X_real)

# Génération
X_copule_norm = copule.sample(n_samples)
df_bayes = decode_synthetic(
    X_copule_norm, scaler, label_encoders,
    COLS_NUMERIQUES, COLS_CATEGORIELLES, ALL_COLS, df_sample
)

print(f'Shape Bayésien (Copule) : {df_bayes.shape}')
print(f'Taux défaut Bayésien   : {df_bayes[CIBLE].mean():.2%}')
print(f'Taux défaut réel       : {df_sample[CIBLE].mean():.2%}')
print('✅ Modèle bayésien (copule) entraîné')

In [ ]:
# Visualisation : heatmap de la matrice de corrélation
plt.figure(figsize=(12, 10))
sns.heatmap(
    copule.corr_matrix,
    xticklabels=ALL_COLS,
    yticklabels=ALL_COLS,
    cmap='coolwarm', center=0, annot=False
)
plt.title('Matrice de corrélation dans l\'espace gaussien (Copule)')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.show()

## 7. Comparaison visuelle des trois modèles

In [ ]:
cols_viz = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AGE_YEARS', 'EXT_SOURCE_2']

fig, axes = plt.subplots(len(cols_viz), 4, figsize=(20, 4 * len(cols_viz)))

for i, col in enumerate(cols_viz):
    datasets = [(df_sample, 'Réel', 'steelblue'),
                (df_ctgan, 'GAN', 'tomato'),
                (df_tvae, 'TVAE', 'green'),
                (df_bayes, 'Bayésien', 'purple')]
    for j, (data, name, color) in enumerate(datasets):
        ax = axes[i, j]
        if col in data.columns:
            vals = pd.to_numeric(data[col], errors='coerce').dropna()
            sns.kdeplot(vals, ax=ax, color=color, fill=True, alpha=0.4)
        ax.set_title(f'{name} — {col}', fontsize=8)
        ax.set_xlabel('')

plt.suptitle('Comparaison des distributions — 3 modèles génératifs', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. Confidentialité from scratch

In [ ]:
# -------------------------------------------------------
# k-ANONYMAT from scratch
# -------------------------------------------------------

def compute_k_anonymity(df, quasi_identifiers):
    groups = df.groupby(quasi_identifiers).size()
    return groups.min(), groups


# -------------------------------------------------------
# l-DIVERSITY from scratch
# -------------------------------------------------------

def compute_l_diversity(df, quasi_identifiers, sensitive_col):
    l_values = df.groupby(quasi_identifiers)[sensitive_col].nunique()
    return l_values.min(), l_values


# -------------------------------------------------------
# DIFFERENTIAL PRIVACY from scratch
# Mécanisme de Laplace : bruit ~ Laplace(0, Δf / ε)
# -------------------------------------------------------

def laplace_mechanism(true_value, sensitivity, epsilon):
    """
    Ajoute du bruit de Laplace à une statistique.
    true_value  : valeur réelle
    sensitivity : sensibilité globale (Δf)
    epsilon     : budget de confidentialité
    """
    scale = sensitivity / epsilon
    noise = np.random.laplace(0, scale)
    return true_value + noise


def gaussian_mechanism(true_value, sensitivity, epsilon, delta=1e-5):
    """
    Mécanisme gaussien (ε, δ)-DP.
    Plus adapté aux vecteurs ou résultats à multi-dimensions.
    """
    sigma = sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon
    noise = np.random.normal(0, sigma)
    return true_value + noise


def apply_dp_to_dataframe(df, numeric_cols, epsilon=1.0):
    """
    Applique le mécanisme de Laplace colonne par colonne
    sur un DataFrame numérique.
    """
    df_dp = df.copy()
    n = len(df_dp)
    for col in numeric_cols:
        col_range = float(df_dp[col].max() - df_dp[col].min())
        sensitivity = col_range / n
        noise = np.random.laplace(0, sensitivity / epsilon, size=n)
        df_dp[col] = df_dp[col].astype(float) + noise
    return df_dp


print('Fonctions de confidentialité définies ✅')

In [ ]:
# Application des métriques
QI = ['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_INCOME_TYPE']

# k-anonymat
k_reel, grp_reel = compute_k_anonymity(df_sample, QI)
k_ctgan, grp_ctgan = compute_k_anonymity(df_ctgan, QI)
k_tvae, grp_tvae = compute_k_anonymity(df_tvae, QI)
k_bayes, grp_bayes = compute_k_anonymity(df_bayes, QI)

# l-diversity
l_reel, _ = compute_l_diversity(df_sample, QI, CIBLE)
l_ctgan, _ = compute_l_diversity(df_ctgan, QI, CIBLE)
l_tvae, _ = compute_l_diversity(df_tvae, QI, CIBLE)
l_bayes, _ = compute_l_diversity(df_bayes, QI, CIBLE)

print('=== RÉSUMÉ CONFIDENTIALITÉ ===')
print(f"{'Modèle':<15} {'k_min':>10} {'l_min':>10}")
print('-' * 38)
for nom, k, l in [('Réel', k_reel, l_reel), ('GAN', k_ctgan, l_ctgan),
                   ('TVAE', k_tvae, l_tvae), ('Bayésien', k_bayes, l_bayes)]:
    print(f"{nom:<15} {k:>10} {l:>10}")

In [ ]:
# Differential Privacy — comparaison statistiques pour différents ε
col_dp = 'AMT_INCOME_TOTAL'
true_mean = df_sample[col_dp].mean()
col_range = df_sample[col_dp].max() - df_sample[col_dp].min()
sensitivity = col_range / len(df_sample)

EPSILON_VALUES = [0.1, 0.5, 1.0, 5.0, 10.0]

print(f'Vraie moyenne {col_dp} : {true_mean:,.0f}')
print()
print(f"{'ε':<8} {'Moyenne DP':>15} {'Erreur relative':>18} {'Bruit scale':>15}")
print('-' * 60)

for eps in EPSILON_VALUES:
    mean_dp = laplace_mechanism(true_mean, sensitivity, eps)
    erreur = abs(mean_dp - true_mean) / true_mean
    scale = sensitivity / eps
    print(f"ε={eps:<5} {mean_dp:>15,.0f} {erreur:>17.4%} {scale:>15.2f}")

In [ ]:
# Application DP aux données synthétiques GAN
COLS_NUM_CLEAN = [c for c in COLS_NUMERIQUES if c in df_ctgan.columns]
df_ctgan_dp = apply_dp_to_dataframe(df_ctgan, COLS_NUM_CLEAN, epsilon=1.0)

print('✅ Données GAN + DP générées (ε=1.0)')

## 9. Évaluation TSTR — Utilité prédictive

In [ ]:
def prepare_X_y(df, num_cols, cat_cols, target):
    df_enc = df.copy()
    le = LabelEncoder()
    for col in cat_cols:
        df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    cols = [c for c in num_cols + cat_cols if c in df_enc.columns]
    X = df_enc[cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
    y = pd.to_numeric(df_enc[target], errors='coerce').fillna(0).values.astype(int)
    return X, y


# Split test réel
df_reel_train, df_reel_test = train_test_split(df_sample, test_size=0.2, random_state=42)

results_auc = {}

print('Calcul des AUC TSTR...')
for nom, df_synth in [('Réel (TRTR)', df_reel_train),
                       ('GAN (TSTR)', df_ctgan),
                       ('TVAE (TSTR)', df_tvae),
                       ('Bayésien (TSTR)', df_bayes)]:
    X_train, y_train = prepare_X_y(df_synth, COLS_NUMERIQUES, COLS_CATEGORIELLES, CIBLE)
    X_test, y_test = prepare_X_y(df_reel_test, COLS_NUMERIQUES, COLS_CATEGORIELLES, CIBLE)
    model = LogisticRegression(max_iter=500, random_state=42)
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    results_auc[nom] = auc
    print(f'  {nom:<20} AUC = {auc:.4f}')

print('\n✅ Évaluation TSTR terminée')

In [ ]:
# Visualisation finale
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart AUC
noms = list(results_auc.keys())
aucs = list(results_auc.values())
colors = ['steelblue', 'tomato', 'green', 'purple']

bars = axes[0].bar(noms, aucs, color=colors)
axes[0].axhline(y=aucs[0], color='navy', linestyle='--', alpha=0.7, label='Baseline réel')
axes[0].set_ylim(0.5, max(aucs) + 0.05)
axes[0].set_title('Comparaison AUC — TSTR')
axes[0].set_ylabel('AUC ROC')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend()
for bar, val in zip(bars, aucs):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

# Tableau comparatif
table_data = [
    ['Réel', f'{k_reel}', f'{l_reel}', f'{aucs[0]:.4f}'],
    ['GAN', f'{k_ctgan}', f'{l_ctgan}', f'{aucs[1]:.4f}'],
    ['TVAE', f'{k_tvae}', f'{l_tvae}', f'{aucs[2]:.4f}'],
    ['Bayésien', f'{k_bayes}', f'{l_bayes}', f'{aucs[3]:.4f}']
]

axes[1].axis('off')
table = axes[1].table(
    cellText=table_data,
    colLabels=['Modèle', 'k-anon', 'l-div', 'AUC TSTR'],
    cellLoc='center', loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.5, 2.2)
axes[1].set_title('Tableau comparatif final', pad=20)

plt.suptitle('Résultats — Modèles génératifs (version sans packages)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Sauvegarde
df_ctgan.to_csv('./sample_data/synthetic_gan_scratch.csv', index=False)
df_tvae.to_csv('./sample_data/synthetic_tvae_scratch.csv', index=False)
df_bayes.to_csv('./sample_data/synthetic_bayes_scratch.csv', index=False)

print('✅ Données sauvegardées :')
print('  - synthetic_gan_scratch.csv')
print('  - synthetic_tvae_scratch.csv')
print('  - synthetic_bayes_scratch.csv')